#### Import libraries

In [ ]:
# Standard Libaries
import sys
from pathlib import Path
import shutil

# Setting Path
notebook_path = Path().resolve()
parent_dir = notebook_path.parent
src_dir = parent_dir / "src"
sys.path.append(str(parent_dir))
sys.path.append(str(src_dir))
import pandas as pd
import numpy as np

from src.dataset import load_image_dataframe
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#### Functions

In [ ]:
def plot_distribution_by_column(train_df: pd.DataFrame, test_df: pd.DataFrame, column: str, label_map: dict = None):
    train_vals = train_df[column].dropna()
    test_vals = test_df[column].dropna()

    if label_map:
        train_vals = train_vals.map(label_map)
        test_vals = test_vals.map(label_map)
        categories = [str(label_map[v]).capitalize() for v in sorted(label_map)]
    else:
        categories_raw = sorted(set(train_vals.unique()) | set(test_vals.unique()))
        categories = [str(cat).capitalize() for cat in categories_raw]
        train_vals = pd.Categorical(train_vals.astype(str).str.capitalize(), categories=categories)
        test_vals = pd.Categorical(test_vals.astype(str).str.capitalize(), categories=categories)

    def get_counts_and_props(values, categories):
        counts = values.value_counts().reindex(categories, fill_value=0)
        props = counts / counts.sum()
        return counts, props

    train_counts, train_props = get_counts_and_props(train_vals, categories)
    test_counts, test_props = get_counts_and_props(test_vals, categories)

    x = range(len(categories))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4))
    bars1 = ax.bar([i - width / 2 for i in x], train_props.values * 100, width, label="Train Set")
    bars2 = ax.bar([i + width / 2 for i in x], test_props.values * 100, width, label="Test Set", color="orange")

    for bars, props, counts in zip([bars1, bars2], [train_props, test_props], [train_counts, test_counts]):
        for bar, prop, count in zip(bars, props, counts):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f"{prop * 100:.1f}%\n(n={count})",
                ha="center",
                va="bottom",
                fontsize=12,
            )

    ax.set_ylabel("Proportion (%)", fontsize=14)
    ax.set_ylim(0, 100)
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=14)
    ax.tick_params(axis="y", labelsize=12)
    ax.legend(fontsize=12)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()

    return fig


def patient_based_split(df: pd.DataFrame, test_size: float = 0.15, random_state: int = 0):
    df = df.copy()

    unique_patients = df["patient_id"].unique()
    train_patients, test_patients = train_test_split(unique_patients, test_size=test_size, random_state=random_state)

    train_df = df[df["patient_id"].isin(train_patients)].reset_index(drop=True)
    test_df = df[df["patient_id"].isin(test_patients)].reset_index(drop=True)

    return train_df, test_df


def copy_split_files(train_df, test_df, dataset_root="split_dataset"):
    def copy_files(df, split_name):
        for _, row in df.iterrows():
            image_src = Path(row["image_path"])
            image_dst = Path(dataset_root) / split_name / "data/Unassigned" / image_src.name

            # Derive annotation file name from image filename
            annotation_name = image_src.with_suffix(".json").name
            annotation_src = Path(dataset_root) / "full_dataset/annotations/classification/Unassigned" / annotation_name
            annotation_dst = Path(dataset_root) / split_name / "annotations/classification/Unassigned" / annotation_name

            # Create target directories
            image_dst.parent.mkdir(parents=True, exist_ok=True)
            annotation_dst.parent.mkdir(parents=True, exist_ok=True)

            # Copy files
            if image_src.exists():
                shutil.copy2(image_src, image_dst)
            else:
                print(f"[WARN] Image missing: {image_src}")

            if annotation_src.exists():
                shutil.copy2(annotation_src, annotation_dst)
            else:
                print(f"[WARN] Annotation missing: {annotation_src}")

    copy_files(train_df, "train")
    copy_files(test_df, "test")


def compute_label_distribution(df, label_col):
    counts = df[label_col].value_counts(normalize=True).sort_index()
    return counts


def find_best_random_state_multi(
    df: pd.DataFrame,
    label_cols: list[str],
    n_trials: int = 1000,
    test_size: float = 0.15,
    patient_col: str = "patient_id",
):
    best_state = None
    best_score = float("inf")
    best_split = None

    patient_ids = df[patient_col].unique()

    for seed in range(n_trials):
        train_ids, test_ids = train_test_split(patient_ids, test_size=test_size, random_state=seed)
        train_df = df[df[patient_col].isin(train_ids)].copy()
        test_df = df[df[patient_col].isin(test_ids)].copy()

        total_dist = 0
        valid = True

        for label_col in label_cols:
            p = compute_label_distribution(train_df, label_col)
            q = compute_label_distribution(test_df, label_col)

            all_labels = sorted(set(p.index).union(q.index))
            p = p.reindex(all_labels, fill_value=0)
            q = q.reindex(all_labels, fill_value=0)

            dist = np.abs(p - q).sum()

            if np.isnan(dist):
                valid = False
                break

            total_dist += dist

        if valid and total_dist < best_score:
            best_score = total_dist
            best_state = seed
            best_split = (train_df.copy(), test_df.copy())

    return best_state, best_score, best_split

#### Data information

In [ ]:
full_dataset = load_image_dataframe(data_path=parent_dir / "data/full_dataset", split="", exclusion=True)
print("------------------------------------------------------")
print(f"Length of dataset: {len(full_dataset)}")
print("Number of unique patients:", full_dataset["patient_id"].nunique())
print("Number of unique images:", full_dataset["image_path"].nunique())
print("Manufacturers:", full_dataset["manufacturer"].unique())
print("Annotators:", full_dataset["annotators"].explode().unique())
print("------------------------------------------------------")

#### Train and test split

In [ ]:
best_seed, score, (train_df, test_df) = find_best_random_state_multi(
    full_dataset, label_cols=["oiq", "expansion", "mucosal_cleaning", "manufacturer"], n_trials=500
)

print(f"Best random_state: {best_seed}")

train_df, test_df = patient_based_split(full_dataset, test_size=0.15, random_state=best_seed)

print(train_df["patient_id"].nunique(), "unique patients in train set")
print(test_df["patient_id"].nunique(), "unique patients in test set")

In [ ]:
label_map = {0: "Poor", 1: "Adequate", 2: "Good"}
label_map_retro = {1: "Retrograde", 0: "Non-retrograde"}

fig_oiq = plot_distribution_by_column(train_df, test_df, column="oiq", label_map=label_map)
# fig_oiq.savefig("oiq_distribution.png", dpi=300, bbox_inches="tight")
fig_expansion = plot_distribution_by_column(train_df, test_df, column="expansion", label_map=label_map)
# fig_expansion.savefig("expansion_distribution.png", dpi=300, bbox_inches="tight")
fig_mucosal_cleaning = plot_distribution_by_column(train_df, test_df, column="mucosal_cleaning", label_map=label_map)
# fig_mucosal_cleaning.savefig("mucosal_cleaning_distribution.png", dpi=300, bbox_inches="tight")
fig_retrograde = plot_distribution_by_column(train_df, test_df, column="retrograde", label_map=label_map_retro)
# fig_retrograde.savefig("retrograde_distribution.png", dpi=300, bbox_inches="tight")
fig_manufacturer = plot_distribution_by_column(train_df, test_df, column="manufacturer")
# fig_manufacturer.savefig("manufacturer_distribution.png", dpi=300, bbox_inches="tight")

#### Create train and test folders

In [ ]:
copy_split_files(train_df, test_df, dataset_root=parent_dir / "data")